# SAP O2C Data Modelling

Schemas are sourced from `docs/sap_o2c_schemas.md` (19 tables, validated against `data/sap-o2c-data/`).


In [ ]:
import pandas as pd
import duckdb
import os
import glob
import magic_duckdb

DATA_DIR = '../data/sap-o2c-data'

con = duckdb.connect('o2c_data.db')

# magic_duckdb setup
magic_duckdb.MAGIC_NAME = 'sql'
%load_ext magic_duckdb
%sql -co con
%config MagicDuckDB.displaylimit = 0

## Schema Creation (DDL)
Creating all 19 tables with proper types from `sap_o2c_schemas.md`.


In [ ]:
ddl = '''
-- Drop and recreate all tables with proper DDL from sap_o2c_schemas.md

CREATE TABLE IF NOT EXISTS billing_document_headers (
    billing_document           VARCHAR(10)    NOT NULL PRIMARY KEY,
    billing_document_type      VARCHAR(4),
    creation_date              DATE,
    creation_time              TIME,
    last_change_datetime       TIMESTAMPTZ,
    billing_document_date      DATE,
    billing_document_is_cancelled BOOLEAN     DEFAULT FALSE,
    cancelled_billing_document VARCHAR(10),
    total_net_amount           NUMERIC(15, 2),
    transaction_currency       VARCHAR(5),
    company_code               VARCHAR(4),
    fiscal_year                VARCHAR(4),
    accounting_document        VARCHAR(10),
    sold_to_party              VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS billing_document_cancellations (
    billing_document              VARCHAR(10)    NOT NULL PRIMARY KEY,
    billing_document_type         VARCHAR(4),
    creation_date                 DATE,
    creation_time                 TIME,
    last_change_datetime          TIMESTAMPTZ,
    billing_document_date         DATE,
    billing_document_is_cancelled BOOLEAN        DEFAULT FALSE,
    cancelled_billing_document    VARCHAR(10),
    total_net_amount              NUMERIC(15, 2),
    transaction_currency          VARCHAR(5),
    company_code                  VARCHAR(4),
    fiscal_year                   VARCHAR(4),
    accounting_document           VARCHAR(10),
    sold_to_party                 VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS billing_document_items (
    billing_document          VARCHAR(10)   NOT NULL,
    billing_document_item     VARCHAR(6)    NOT NULL,
    material                  VARCHAR(40),
    billing_quantity          NUMERIC(15, 3),
    billing_quantity_unit     VARCHAR(3),
    net_amount                NUMERIC(15, 2),
    transaction_currency      VARCHAR(5),
    reference_sd_document     VARCHAR(10),
    reference_sd_document_item VARCHAR(6),
    PRIMARY KEY (billing_document, billing_document_item)
);

CREATE TABLE IF NOT EXISTS business_partner_addresses (
    business_partner           VARCHAR(10)   NOT NULL,
    address_id                 VARCHAR(10)   NOT NULL,
    validity_start_date        DATE,
    validity_end_date          DATE,
    address_uuid               VARCHAR(36),
    address_time_zone          VARCHAR(10),
    city_name                  VARCHAR(40),
    country                    VARCHAR(3),
    po_box                     VARCHAR(10),
    po_box_deviating_city_name VARCHAR(40),
    po_box_deviating_country   VARCHAR(3),
    po_box_deviating_region    VARCHAR(3),
    po_box_is_without_number   BOOLEAN       DEFAULT FALSE,
    po_box_lobby_name          VARCHAR(40),
    po_box_postal_code         VARCHAR(10),
    postal_code                VARCHAR(10),
    region                     VARCHAR(3),
    street_name                VARCHAR(60),
    tax_jurisdiction           VARCHAR(15),
    transport_zone             VARCHAR(10),
    PRIMARY KEY (business_partner, address_id)
);

CREATE TABLE IF NOT EXISTS business_partners (
    business_partner           VARCHAR(10)   NOT NULL PRIMARY KEY,
    customer                   VARCHAR(10),
    business_partner_category  VARCHAR(1),
    business_partner_full_name VARCHAR(81),
    business_partner_grouping  VARCHAR(4),
    business_partner_name      VARCHAR(81),
    correspondence_language    VARCHAR(2),
    created_by_user            VARCHAR(12),
    creation_date              DATE,
    creation_time              TIME,
    first_name                 VARCHAR(40),
    form_of_address            VARCHAR(4),
    industry                   VARCHAR(10),
    last_change_date           DATE,
    last_name                  VARCHAR(40),
    organization_bp_name1      VARCHAR(40),
    organization_bp_name2      VARCHAR(40),
    business_partner_is_blocked BOOLEAN      DEFAULT FALSE,
    is_marked_for_archiving    BOOLEAN       DEFAULT FALSE
);

CREATE TABLE IF NOT EXISTS customer_company_assignments (
    customer                         VARCHAR(10)   NOT NULL,
    company_code                     VARCHAR(4)    NOT NULL,
    accounting_clerk                 VARCHAR(2),
    accounting_clerk_fax_number      VARCHAR(31),
    accounting_clerk_internet_address VARCHAR(130),
    accounting_clerk_phone_number    VARCHAR(30),
    alternative_payer_account        VARCHAR(10),
    payment_blocking_reason          VARCHAR(1),
    payment_methods_list             VARCHAR(10),
    payment_terms                    VARCHAR(4),
    reconciliation_account           VARCHAR(10),
    deletion_indicator               BOOLEAN       DEFAULT FALSE,
    customer_account_group           VARCHAR(4),
    PRIMARY KEY (customer, company_code)
);

CREATE TABLE IF NOT EXISTS customer_sales_area_assignments (
    customer                      VARCHAR(10)   NOT NULL,
    sales_organization            VARCHAR(4)    NOT NULL,
    distribution_channel          VARCHAR(2)    NOT NULL,
    division                      VARCHAR(2)    NOT NULL,
    billing_is_blocked_for_customer VARCHAR(2),
    complete_delivery_is_defined  BOOLEAN       DEFAULT FALSE,
    credit_control_area           VARCHAR(4),
    currency                      VARCHAR(5),
    customer_payment_terms        VARCHAR(4),
    delivery_priority             VARCHAR(2),
    incoterms_classification      VARCHAR(3),
    incoterms_location1           VARCHAR(70),
    sales_group                   VARCHAR(3),
    sales_office                  VARCHAR(4),
    shipping_condition            VARCHAR(2),
    sls_unlmtd_ovrdeliv_is_allwd  BOOLEAN       DEFAULT FALSE,
    supplying_plant               VARCHAR(4),
    sales_district                VARCHAR(6),
    exchange_rate_type            VARCHAR(4),
    PRIMARY KEY (customer, sales_organization, distribution_channel, division)
);

CREATE TABLE IF NOT EXISTS journal_entry_items_accounts_receivable (
    company_code                   VARCHAR(4)    NOT NULL,
    fiscal_year                    VARCHAR(4)    NOT NULL,
    accounting_document            VARCHAR(10)   NOT NULL,
    accounting_document_item       VARCHAR(3)    NOT NULL,
    gl_account                     VARCHAR(10),
    reference_document             VARCHAR(16),
    cost_center                    VARCHAR(10),
    profit_center                  VARCHAR(10),
    transaction_currency           VARCHAR(5),
    amount_in_transaction_currency NUMERIC(23, 2),
    company_code_currency          VARCHAR(5),
    amount_in_company_code_currency NUMERIC(23, 2),
    posting_date                   DATE,
    document_date                  DATE,
    accounting_document_type       VARCHAR(2),
    assignment_reference           VARCHAR(18),
    last_change_datetime           TIMESTAMPTZ,
    customer                       VARCHAR(10),
    financial_account_type         VARCHAR(1),
    clearing_date                  DATE,
    clearing_accounting_document   VARCHAR(10),
    clearing_doc_fiscal_year       VARCHAR(4),
    PRIMARY KEY (company_code, fiscal_year, accounting_document, accounting_document_item)
);

CREATE TABLE IF NOT EXISTS outbound_delivery_headers (
    delivery_document               VARCHAR(10)   NOT NULL PRIMARY KEY,
    actual_goods_movement_date      DATE,
    actual_goods_movement_time      TIME,
    creation_date                   DATE,
    creation_time                   TIME,
    delivery_block_reason           VARCHAR(2),
    hdr_general_incompletion_status VARCHAR(1),
    header_billing_block_reason     VARCHAR(2),
    last_change_date                DATE,
    overall_goods_movement_status   VARCHAR(1),
    overall_picking_status          VARCHAR(1),
    overall_proof_of_delivery_status VARCHAR(1),
    shipping_point                  VARCHAR(4)
);

CREATE TABLE IF NOT EXISTS outbound_delivery_items (
    delivery_document          VARCHAR(10)   NOT NULL,
    delivery_document_item     VARCHAR(6)    NOT NULL,
    actual_delivery_quantity   NUMERIC(15, 3),
    batch                      VARCHAR(10),
    delivery_quantity_unit     VARCHAR(3),
    item_billing_block_reason  VARCHAR(2),
    last_change_date           DATE,
    plant                      VARCHAR(4),
    reference_sd_document      VARCHAR(10),
    reference_sd_document_item VARCHAR(6),
    storage_location           VARCHAR(4),
    PRIMARY KEY (delivery_document, delivery_document_item)
);

CREATE TABLE IF NOT EXISTS payments_accounts_receivable (
    company_code                    VARCHAR(4)    NOT NULL,
    fiscal_year                     VARCHAR(4)    NOT NULL,
    accounting_document             VARCHAR(10)   NOT NULL,
    accounting_document_item        VARCHAR(3)    NOT NULL,
    clearing_date                   DATE,
    clearing_accounting_document    VARCHAR(10),
    clearing_doc_fiscal_year        VARCHAR(4),
    amount_in_transaction_currency  NUMERIC(23, 2),
    transaction_currency            VARCHAR(5),
    amount_in_company_code_currency NUMERIC(23, 2),
    company_code_currency           VARCHAR(5),
    customer                        VARCHAR(10),
    invoice_reference               VARCHAR(10),
    invoice_reference_fiscal_year   VARCHAR(4),
    sales_document                  VARCHAR(10),
    sales_document_item             VARCHAR(6),
    posting_date                    DATE,
    document_date                   DATE,
    assignment_reference            VARCHAR(18),
    gl_account                      VARCHAR(10),
    financial_account_type          VARCHAR(1),
    profit_center                   VARCHAR(10),
    cost_center                     VARCHAR(10),
    PRIMARY KEY (company_code, fiscal_year, accounting_document, accounting_document_item)
);

CREATE TABLE IF NOT EXISTS plants (
    plant                             VARCHAR(4)    NOT NULL PRIMARY KEY,
    plant_name                        VARCHAR(30),
    valuation_area                    VARCHAR(4),
    plant_customer                    VARCHAR(10),
    plant_supplier                    VARCHAR(10),
    factory_calendar                  VARCHAR(2),
    default_purchasing_organization   VARCHAR(4),
    sales_organization                VARCHAR(4),
    address_id                        VARCHAR(10),
    plant_category                    VARCHAR(1),
    distribution_channel              VARCHAR(2),
    division                          VARCHAR(2),
    language                          VARCHAR(2),
    is_marked_for_archiving           BOOLEAN       DEFAULT FALSE
);

CREATE TABLE IF NOT EXISTS product_descriptions (
    product             VARCHAR(40)   NOT NULL,
    language            VARCHAR(2)    NOT NULL,
    product_description VARCHAR(40),
    PRIMARY KEY (product, language)
);

CREATE TABLE IF NOT EXISTS product_plants (
    product                      VARCHAR(40)   NOT NULL,
    plant                        VARCHAR(4)    NOT NULL,
    country_of_origin            VARCHAR(3),
    region_of_origin             VARCHAR(3),
    production_invtry_managed_loc VARCHAR(4),
    availability_check_type      VARCHAR(2),
    fiscal_year_variant          VARCHAR(2),
    profit_center                VARCHAR(10),
    mrp_type                     VARCHAR(2),
    PRIMARY KEY (product, plant)
);

CREATE TABLE IF NOT EXISTS product_storage_locations (
    product                               VARCHAR(40)   NOT NULL,
    plant                                 VARCHAR(4)    NOT NULL,
    storage_location                      VARCHAR(4)    NOT NULL,
    physical_inventory_block_ind          VARCHAR(1),
    date_of_last_posted_cnt_un_rstrcd_stk DATE,
    PRIMARY KEY (product, plant, storage_location)
);

CREATE TABLE IF NOT EXISTS products (
    product                          VARCHAR(40)   NOT NULL PRIMARY KEY,
    product_type                     VARCHAR(4),
    cross_plant_status               VARCHAR(2),
    cross_plant_status_validity_date DATE,
    creation_date                    DATE,
    created_by_user                  VARCHAR(12),
    last_change_date                 DATE,
    last_change_datetime             TIMESTAMPTZ,
    is_marked_for_deletion           BOOLEAN       DEFAULT FALSE,
    product_old_id                   VARCHAR(40),
    gross_weight                     NUMERIC(13, 3),
    weight_unit                      VARCHAR(3),
    net_weight                       NUMERIC(13, 3),
    product_group                    VARCHAR(9),
    base_unit                        VARCHAR(3),
    division                         VARCHAR(2),
    industry_sector                  VARCHAR(1)
);

CREATE TABLE IF NOT EXISTS sales_order_headers (
    sales_order                     VARCHAR(10)   NOT NULL PRIMARY KEY,
    sales_order_type                VARCHAR(4),
    sales_organization              VARCHAR(4),
    distribution_channel            VARCHAR(2),
    organization_division           VARCHAR(2),
    sales_group                     VARCHAR(3),
    sales_office                    VARCHAR(4),
    sold_to_party                   VARCHAR(10),
    creation_date                   DATE,
    created_by_user                 VARCHAR(12),
    last_change_datetime            TIMESTAMPTZ,
    total_net_amount                NUMERIC(15, 2),
    overall_delivery_status         VARCHAR(1),
    overall_ord_reltd_billg_status  VARCHAR(1),
    overall_sd_doc_reference_status VARCHAR(1),
    transaction_currency            VARCHAR(5),
    pricing_date                    DATE,
    requested_delivery_date         DATE,
    header_billing_block_reason     VARCHAR(2),
    delivery_block_reason           VARCHAR(2),
    incoterms_classification        VARCHAR(3),
    incoterms_location1             VARCHAR(70),
    customer_payment_terms          VARCHAR(4),
    total_credit_check_status       VARCHAR(1)
);

CREATE TABLE IF NOT EXISTS sales_order_items (
    sales_order                VARCHAR(10)   NOT NULL,
    sales_order_item           VARCHAR(6)    NOT NULL,
    sales_order_item_category  VARCHAR(4),
    material                   VARCHAR(40),
    requested_quantity         NUMERIC(15, 3),
    requested_quantity_unit    VARCHAR(3),
    transaction_currency       VARCHAR(5),
    net_amount                 NUMERIC(15, 2),
    material_group             VARCHAR(9),
    production_plant           VARCHAR(4),
    storage_location           VARCHAR(4),
    sales_document_rjcn_reason VARCHAR(2),
    item_billing_block_reason  VARCHAR(2),
    PRIMARY KEY (sales_order, sales_order_item)
);

CREATE TABLE IF NOT EXISTS sales_order_schedule_lines (
    sales_order                           VARCHAR(10)   NOT NULL,
    sales_order_item                      VARCHAR(6)    NOT NULL,
    schedule_line                         VARCHAR(4)    NOT NULL,
    confirmed_delivery_date               DATE,
    order_quantity_unit                   VARCHAR(3),
    confd_order_qty_by_matl_avail_check   NUMERIC(15, 3),
    PRIMARY KEY (sales_order, sales_order_item, schedule_line)
);
'''

# Execute each statement individually
for stmt in [s.strip() for s in ddl.split(';') if s.strip()]:
    con.execute(stmt)

tables = con.execute("SELECT table_name FROM duckdb_tables() ORDER BY table_name").fetchdf()
print(f'Created {len(tables)} tables:')
print(tables['table_name'].tolist())

## Data Loading
Loads all JSONL files using explicit camelCase→snake_case column mapping.
All values are read as VARCHAR first, then cast via `NULLIF(TRIM(...), '')` to handle empty strings and NULLs cleanly.
Nested `creationTime` / `actualGoodsMovementTime` objects are excluded from initial load (DuckDB reads the object as-is).


In [ ]:
import os
import glob

DATA_DIR = '../data/sap-o2c-data'

TABLE_COLUMN_MAPS = {'billing_document_headers': {'billingDocument': 'billing_document', 'billingDocumentType': 'billing_document_type', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'lastChangeDateTime': 'last_change_datetime', 'billingDocumentDate': 'billing_document_date', 'billingDocumentIsCancelled': 'billing_document_is_cancelled', 'cancelledBillingDocument': 'cancelled_billing_document', 'totalNetAmount': 'total_net_amount', 'transactionCurrency': 'transaction_currency', 'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'soldToParty': 'sold_to_party'}, 'billing_document_cancellations': {'billingDocument': 'billing_document', 'billingDocumentType': 'billing_document_type', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'lastChangeDateTime': 'last_change_datetime', 'billingDocumentDate': 'billing_document_date', 'billingDocumentIsCancelled': 'billing_document_is_cancelled', 'cancelledBillingDocument': 'cancelled_billing_document', 'totalNetAmount': 'total_net_amount', 'transactionCurrency': 'transaction_currency', 'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'soldToParty': 'sold_to_party'}, 'billing_document_items': {'billingDocument': 'billing_document', 'billingDocumentItem': 'billing_document_item', 'material': 'material', 'billingQuantity': 'billing_quantity', 'billingQuantityUnit': 'billing_quantity_unit', 'netAmount': 'net_amount', 'transactionCurrency': 'transaction_currency', 'referenceSdDocument': 'reference_sd_document', 'referenceSdDocumentItem': 'reference_sd_document_item'}, 'business_partner_addresses': {'businessPartner': 'business_partner', 'addressId': 'address_id', 'validityStartDate': 'validity_start_date', 'validityEndDate': 'validity_end_date', 'addressUuid': 'address_uuid', 'addressTimeZone': 'address_time_zone', 'cityName': 'city_name', 'country': 'country', 'poBox': 'po_box', 'poBoxDeviatingCityName': 'po_box_deviating_city_name', 'poBoxDeviatingCountry': 'po_box_deviating_country', 'poBoxDeviatingRegion': 'po_box_deviating_region', 'poBoxIsWithoutNumber': 'po_box_is_without_number', 'poBoxLobbyName': 'po_box_lobby_name', 'poBoxPostalCode': 'po_box_postal_code', 'postalCode': 'postal_code', 'region': 'region', 'streetName': 'street_name', 'taxJurisdiction': 'tax_jurisdiction', 'transportZone': 'transport_zone'}, 'business_partners': {'businessPartner': 'business_partner', 'customer': 'customer', 'businessPartnerCategory': 'business_partner_category', 'businessPartnerFullName': 'business_partner_full_name', 'businessPartnerGrouping': 'business_partner_grouping', 'businessPartnerName': 'business_partner_name', 'correspondenceLanguage': 'correspondence_language', 'createdByUser': 'created_by_user', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'firstName': 'first_name', 'formOfAddress': 'form_of_address', 'industry': 'industry', 'lastChangeDate': 'last_change_date', 'lastName': 'last_name', 'organizationBpName1': 'organization_bp_name1', 'organizationBpName2': 'organization_bp_name2', 'businessPartnerIsBlocked': 'business_partner_is_blocked', 'isMarkedForArchiving': 'is_marked_for_archiving'}, 'customer_company_assignments': {'customer': 'customer', 'companyCode': 'company_code', 'accountingClerk': 'accounting_clerk', 'accountingClerkFaxNumber': 'accounting_clerk_fax_number', 'accountingClerkInternetAddress': 'accounting_clerk_internet_address', 'accountingClerkPhoneNumber': 'accounting_clerk_phone_number', 'alternativePayerAccount': 'alternative_payer_account', 'paymentBlockingReason': 'payment_blocking_reason', 'paymentMethodsList': 'payment_methods_list', 'paymentTerms': 'payment_terms', 'reconciliationAccount': 'reconciliation_account', 'deletionIndicator': 'deletion_indicator', 'customerAccountGroup': 'customer_account_group'}, 'customer_sales_area_assignments': {'customer': 'customer', 'salesOrganization': 'sales_organization', 'distributionChannel': 'distribution_channel', 'division': 'division', 'billingIsBlockedForCustomer': 'billing_is_blocked_for_customer', 'completeDeliveryIsDefined': 'complete_delivery_is_defined', 'creditControlArea': 'credit_control_area', 'currency': 'currency', 'customerPaymentTerms': 'customer_payment_terms', 'deliveryPriority': 'delivery_priority', 'incotermsClassification': 'incoterms_classification', 'incotermsLocation1': 'incoterms_location1', 'salesGroup': 'sales_group', 'salesOffice': 'sales_office', 'shippingCondition': 'shipping_condition', 'slsUnlmtdOvrdelivIsAllwd': 'sls_unlmtd_ovrdeliv_is_allwd', 'supplyingPlant': 'supplying_plant', 'salesDistrict': 'sales_district', 'exchangeRateType': 'exchange_rate_type'}, 'journal_entry_items_accounts_receivable': {'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'glAccount': 'gl_account', 'referenceDocument': 'reference_document', 'costCenter': 'cost_center', 'profitCenter': 'profit_center', 'transactionCurrency': 'transaction_currency', 'amountInTransactionCurrency': 'amount_in_transaction_currency', 'companyCodeCurrency': 'company_code_currency', 'amountInCompanyCodeCurrency': 'amount_in_company_code_currency', 'postingDate': 'posting_date', 'documentDate': 'document_date', 'accountingDocumentType': 'accounting_document_type', 'accountingDocumentItem': 'accounting_document_item', 'assignmentReference': 'assignment_reference', 'lastChangeDateTime': 'last_change_datetime', 'customer': 'customer', 'financialAccountType': 'financial_account_type', 'clearingDate': 'clearing_date', 'clearingAccountingDocument': 'clearing_accounting_document', 'clearingDocFiscalYear': 'clearing_doc_fiscal_year'}, 'outbound_delivery_headers': {'actualGoodsMovementDate': 'actual_goods_movement_date', 'actualGoodsMovementTime': 'actual_goods_movement_time', 'creationDate': 'creation_date', 'creationTime': 'creation_time', 'deliveryBlockReason': 'delivery_block_reason', 'deliveryDocument': 'delivery_document', 'hdrGeneralIncompletionStatus': 'hdr_general_incompletion_status', 'headerBillingBlockReason': 'header_billing_block_reason', 'lastChangeDate': 'last_change_date', 'overallGoodsMovementStatus': 'overall_goods_movement_status', 'overallPickingStatus': 'overall_picking_status', 'overallProofOfDeliveryStatus': 'overall_proof_of_delivery_status', 'shippingPoint': 'shipping_point'}, 'outbound_delivery_items': {'actualDeliveryQuantity': 'actual_delivery_quantity', 'batch': 'batch', 'deliveryDocument': 'delivery_document', 'deliveryDocumentItem': 'delivery_document_item', 'deliveryQuantityUnit': 'delivery_quantity_unit', 'itemBillingBlockReason': 'item_billing_block_reason', 'lastChangeDate': 'last_change_date', 'plant': 'plant', 'referenceSdDocument': 'reference_sd_document', 'referenceSdDocumentItem': 'reference_sd_document_item', 'storageLocation': 'storage_location'}, 'payments_accounts_receivable': {'companyCode': 'company_code', 'fiscalYear': 'fiscal_year', 'accountingDocument': 'accounting_document', 'accountingDocumentItem': 'accounting_document_item', 'clearingDate': 'clearing_date', 'clearingAccountingDocument': 'clearing_accounting_document', 'clearingDocFiscalYear': 'clearing_doc_fiscal_year', 'amountInTransactionCurrency': 'amount_in_transaction_currency', 'transactionCurrency': 'transaction_currency', 'amountInCompanyCodeCurrency': 'amount_in_company_code_currency', 'companyCodeCurrency': 'company_code_currency', 'customer': 'customer', 'invoiceReference': 'invoice_reference', 'invoiceReferenceFiscalYear': 'invoice_reference_fiscal_year', 'salesDocument': 'sales_document', 'salesDocumentItem': 'sales_document_item', 'postingDate': 'posting_date', 'documentDate': 'document_date', 'assignmentReference': 'assignment_reference', 'glAccount': 'gl_account', 'financialAccountType': 'financial_account_type', 'profitCenter': 'profit_center', 'costCenter': 'cost_center'}, 'plants': {'plant': 'plant', 'plantName': 'plant_name', 'valuationArea': 'valuation_area', 'plantCustomer': 'plant_customer', 'plantSupplier': 'plant_supplier', 'factoryCalendar': 'factory_calendar', 'defaultPurchasingOrganization': 'default_purchasing_organization', 'salesOrganization': 'sales_organization', 'addressId': 'address_id', 'plantCategory': 'plant_category', 'distributionChannel': 'distribution_channel', 'division': 'division', 'language': 'language', 'isMarkedForArchiving': 'is_marked_for_archiving'}, 'product_descriptions': {'product': 'product', 'language': 'language', 'productDescription': 'product_description'}, 'product_plants': {'product': 'product', 'plant': 'plant', 'countryOfOrigin': 'country_of_origin', 'regionOfOrigin': 'region_of_origin', 'productionInvtryManagedLoc': 'production_invtry_managed_loc', 'availabilityCheckType': 'availability_check_type', 'fiscalYearVariant': 'fiscal_year_variant', 'profitCenter': 'profit_center', 'mrpType': 'mrp_type'}, 'product_storage_locations': {'product': 'product', 'plant': 'plant', 'storageLocation': 'storage_location', 'physicalInventoryBlockInd': 'physical_inventory_block_ind', 'dateOfLastPostedCntUnRstrcdStk': 'date_of_last_posted_cnt_un_rstrcd_stk'}, 'products': {'product': 'product', 'productType': 'product_type', 'crossPlantStatus': 'cross_plant_status', 'crossPlantStatusValidityDate': 'cross_plant_status_validity_date', 'creationDate': 'creation_date', 'createdByUser': 'created_by_user', 'lastChangeDate': 'last_change_date', 'lastChangeDateTime': 'last_change_datetime', 'isMarkedForDeletion': 'is_marked_for_deletion', 'productOldId': 'product_old_id', 'grossWeight': 'gross_weight', 'weightUnit': 'weight_unit', 'netWeight': 'net_weight', 'productGroup': 'product_group', 'baseUnit': 'base_unit', 'division': 'division', 'industrySector': 'industry_sector'}, 'sales_order_headers': {'salesOrder': 'sales_order', 'salesOrderType': 'sales_order_type', 'salesOrganization': 'sales_organization', 'distributionChannel': 'distribution_channel', 'organizationDivision': 'organization_division', 'salesGroup': 'sales_group', 'salesOffice': 'sales_office', 'soldToParty': 'sold_to_party', 'creationDate': 'creation_date', 'createdByUser': 'created_by_user', 'lastChangeDateTime': 'last_change_datetime', 'totalNetAmount': 'total_net_amount', 'overallDeliveryStatus': 'overall_delivery_status', 'overallOrdReltdBillgStatus': 'overall_ord_reltd_billg_status', 'overallSdDocReferenceStatus': 'overall_sd_doc_reference_status', 'transactionCurrency': 'transaction_currency', 'pricingDate': 'pricing_date', 'requestedDeliveryDate': 'requested_delivery_date', 'headerBillingBlockReason': 'header_billing_block_reason', 'deliveryBlockReason': 'delivery_block_reason', 'incotermsClassification': 'incoterms_classification', 'incotermsLocation1': 'incoterms_location1', 'customerPaymentTerms': 'customer_payment_terms', 'totalCreditCheckStatus': 'total_credit_check_status'}, 'sales_order_items': {'salesOrder': 'sales_order', 'salesOrderItem': 'sales_order_item', 'salesOrderItemCategory': 'sales_order_item_category', 'material': 'material', 'requestedQuantity': 'requested_quantity', 'requestedQuantityUnit': 'requested_quantity_unit', 'transactionCurrency': 'transaction_currency', 'netAmount': 'net_amount', 'materialGroup': 'material_group', 'productionPlant': 'production_plant', 'storageLocation': 'storage_location', 'salesDocumentRjcnReason': 'sales_document_rjcn_reason', 'itemBillingBlockReason': 'item_billing_block_reason'}, 'sales_order_schedule_lines': {'salesOrder': 'sales_order', 'salesOrderItem': 'sales_order_item', 'scheduleLine': 'schedule_line', 'confirmedDeliveryDate': 'confirmed_delivery_date', 'orderQuantityUnit': 'order_quantity_unit', 'confdOrderQtyByMatlAvailCheck': 'confd_order_qty_by_matl_avail_check'}}

NESTED_TIME_JSON_KEYS = {'billing_document_headers': ['creationTime'], 'billing_document_cancellations': ['creationTime'], 'business_partners': ['creationTime'], 'outbound_delivery_headers': ['creationTime', 'actualGoodsMovementTime']}

for table_name, col_map in TABLE_COLUMN_MAPS.items():
    path_pattern = os.path.join(DATA_DIR, table_name, '*.jsonl')
    files = glob.glob(path_pattern)
    if not files:
        print(f"WARNING: no files for {table_name}, skipping")
        continue

    nested_keys = NESTED_TIME_JSON_KEYS.get(table_name, [])
    # Only include non-nested keys in read_json columns spec
    readable_cols = {k: v for k, v in col_map.items() if k not in nested_keys}

    col_spec = ", ".join([f"'{k}': 'VARCHAR'" for k in readable_cols])
    
    # Build SELECT clause: cast each column to target name
    selects = []
    for json_key, sql_col in readable_cols.items():
        selects.append(f"NULLIF(TRIM(\"{json_key}\"), '') AS {sql_col}")
    
    select_clause = ",\n        ".join(selects)
    
    query = f"""
    INSERT INTO {table_name}
        ({", ".join(readable_cols.values())})
    SELECT
        {select_clause}
    FROM read_json_auto('{path_pattern}', columns={{{col_spec}}})
    """
    con.execute(query)
    print(f"Loaded {table_name}")

print("\nAll tables loaded.")


## Verification


In [ ]:
%%sql
SELECT table_name, estimated_size AS row_count
FROM duckdb_tables()
ORDER BY table_name;


In [ ]:
%%sql

SELECT * FROM sales_order_headers LIMIT 5;
